# VŠE FIS Bachelor Thesis Scraper

Main point of this scraper is to scrape data for semestral work for Text analysis. Main requieremnts, the scraped thesis needs to have and with explanations why:
- From **FIS** (Fakulta informatiky a statistiky) | Data for Vice-Dean for Research at FIS (not other faculties) 
- Written in **English** | To work better with corpus and can be applied internationally 
- Submitted **2024 or later** | To be up to date with latest trends, and trying to get as close to todays research direction as possible

This will be the corpus, I will be working with for the analysis.

## Libraries (Focusing on retrieving documents from website)

In [16]:
import os
import re
import time
import random
import logging
from pathlib import Path
from datetime import datetime
from typing import Optional        
from urllib.parse import urljoin, urlencode

import requests
from bs4 import BeautifulSoup        
from tqdm.notebook import tqdm   
import pandas as pd             

## Configuration

In [17]:
BASE_URL      = "https://vskp.vse.cz/"
THESIS_TYPE   = "Bakalářská práce"
TARGET_YEARS  = ["2020", "2021", "2022", "2023", "2024", "2025", "2026"]          # listing pages to scrape

# Detail-page filters
FACULTY_MATCH   = "Fakulta informatiky a statistiky"
LANGUAGE        = "English"
SUBMITTED_AFTER = datetime(2020, 1, 1)

DOWNLOAD_DIR  = Path("downloaded_theses")
METADATA_CSV  = Path("theses_metadata.csv")

DELAY_MIN, DELAY_MAX = 0.001, 0.01     

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36",
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
    "Accept-Language": "cs-CZ,cs;q=0.9,en;q=0.8",
    "Referer": "https://vskp.vse.cz/",
}

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    datefmt="%H:%M:%S",
)
log = logging.getLogger(__name__)

## Helper functions

In [18]:
def polite_sleep():
    time.sleep(random.uniform(DELAY_MIN, DELAY_MAX))


def sanitize_filename(name: str, max_length: int = 120) -> str:
    name = re.sub(r'[\\/*?:"<>|]', "_", name)
    name = re.sub(r"[\s_]+", "_", name).strip("_")
    return name[:max_length]


def parse_czech_date(date_str: str) -> Optional[datetime]:
    """Parse dates like '18. 9. 2024' or '5.10.2025' into datetime objects."""
    date_str = re.sub(r"\s*\.\s*", ".", date_str.strip())
    for fmt in ("%d.%m.%Y", "%d.%m.%y"):
        try:
            return datetime.strptime(date_str, fmt)
        except ValueError:
            pass
    return None


def extract_table_value(soup: BeautifulSoup, label: str) -> str:
    """Find a <th> containing label and return the text of the sibling <td>."""
    for th in soup.find_all("th"):
        if label.lower() in th.get_text(strip=True).lower():
            sibling = th.find_next_sibling("td")
            if sibling:
                return sibling.get_text(strip=True)
    return ""

## Stage 1 — Collect thesis URLs from listing pages

In [19]:
def build_listing_url(year: str, page: int) -> str:
    """
    Construct a listing URL for a specific year and page.

    Args:
        year (str): The year used to filter thesis listings.
        page (int): The page number of the listing results.

    Returns:
        str: A fully constructed URL with encoded query parameters.
    """
    
    params = {
        "title": "", "author": "", "abstract": "",
        "type": THESIS_TYPE, "programme": "",
        "year": year, "page": page,
    }
    return BASE_URL + "?" + urlencode(params)


def get_total_pages(session: requests.Session, year: str) -> int:
    """
    Determine the total number of listing pages available for a given year.

    This function sends a request to the first listing page and attempts
    to extract the total number of pages either from a textual pattern
    (e.g., 'strana 1 / N') or by parsing pagination links.

    Args:
        session (requests.Session): Active HTTP session for making requests.
        year (str): The year used to filter thesis listings.

    Returns:
        int: Total number of pages available for the given year.
             Defaults to 1 if pagination cannot be determined.
    """
    
    resp = session.get(build_listing_url(year, 1), headers=HEADERS, timeout=15)
    resp.raise_for_status()
    soup = BeautifulSoup(resp.text, "html.parser")   #html.parser
    m = re.search(r"strana\s+1\s*/\s*(\d+)", soup.get_text())
    if m:
        return int(m.group(1))
    pages = [
        int(re.search(r"page=(\d+)", a["href"]).group(1))
        for a in soup.select("a[href*='page=']") 
        if re.search(r"page=(\d+)", a.get("href", ""))
    ]
    return max(pages) if pages else 1


def scrape_listing_page(session: requests.Session, year: str, page: int) -> list:
    """
    Scrape thesis listings from a single results page.

    Extracts title, author, and detail URL for each thesis entry
    found in the page's table structure.

    Args:
        session (requests.Session): Active HTTP session for making requests.
        year (str): The year used to filter thesis listings.
        page (int): The page number to scrape.

    Returns:
        list: A list of dictionaries, each containing:
              - listing_year (str)
              - title (str)
              - author (str)
              - detail_url (str)
    """
    
    resp = session.get(build_listing_url(year, page), headers=HEADERS, timeout=15)
    resp.raise_for_status()
    soup = BeautifulSoup(resp.text, "html.parser")   #html.parser

    results = []
    table = soup.find("table")
    if not table:
        return results

    for row in table.find_all("tr")[1:]:
        cells = row.find_all("td")
        if not cells:
            continue
        link_tag = cells[0].find("a")
        if not link_tag:
            continue
        title      = link_tag.get_text(strip=True)
        detail_url = urljoin(BASE_URL, link_tag["href"])
        full_text  = cells[0].get_text(separator="\n", strip=True)
        author     = full_text.replace(title, "").replace(",", "").strip().lower()
        results.append({"listing_year": year, "title": title,
                         "author": author, "detail_url": detail_url})
    return results


def collect_all_listings(session: requests.Session) -> list:
    """
    Collect thesis listings across all target years and pages.

    Iterates over predefined years, determines the number of pages
    for each year, and aggregates results from all listing pages.

    Args:
        session (requests.Session): Active HTTP session for making requests.

    Returns:
        list: A combined list of all scraped thesis entries across years.
    """
    
    all_entries = []
    for year in TARGET_YEARS:
        total = get_total_pages(session, year)
        log.info(f"Year {year}: {total} listing pages found.")
        for page in tqdm(range(1, total + 1), desc=f"Listing {year}"):
            all_entries.extend(scrape_listing_page(session, year, page))
            polite_sleep()
    log.info(f"Raw entries collected: {len(all_entries)}")
    return all_entries

## Stage 2 — Visit detail pages and filter

In [20]:
def fetch_thesis_details(session: requests.Session, detail_url: str) -> Optional[dict]:
    """
    Fetch and parse a thesis detail page, returning structured metadata
    if the entry satisfies all filtering criteria.

    The function performs an HTTP request to the given detail URL, parses
    the HTML content, and extracts relevant thesis metadata. It applies
    multiple filters (language, faculty, and submission date), and returns
    None if any condition is not met or if the request fails.

    Args:
        session (requests.Session): Active HTTP session used for making requests.
        detail_url (str): URL of the thesis detail page.

    Returns:
        Optional[dict]: A dictionary containing extracted thesis metadata if
        all filters pass, otherwise None.

        Returned dictionary structure:
            {
                "language": str,     # Language of the thesis (lowercased)
                "faculty": str,      # Faculty name (lowercased)
                "programme": str,    # Study programme (lowercased)
                "department": str,   # Department (lowercased)
                "title_eng" : str,   # Title of the work (lowercased)
                "submitted": str,    # Raw submission date string
                "defended": str,     # Raw defense date string (lowercased)
                "abstract": str,     # Abstract text (lowercased)
                "key_words": str,    # Key words of the work (lowercased)
                "pdf_url": Optional[str]  # Direct PDF URL if found, else None
            }

    Raises:
        None: All exceptions are handled internally. Network and HTTP errors
        are logged and result in returning None.

    Notes:
        - Applies the following filters:
            1. Language must match `LANGUAGE`.
            2. Faculty must match `FACULTY_MATCH`.
            3. Submission date must be parsed successfully and be after `SUBMITTED_AFTER`.
        - Assumes the page uses UTF-8 encoding.
        - PDF URL is constructed from a detected thesis ID in matching links.
    """
    
    resp = None
    try:
        resp = session.get(detail_url, headers=HEADERS, timeout=15)
        resp.raise_for_status()
    except requests.HTTPError as exc:
        if resp is not None and resp.status_code == 403:
            log.error(f"BLOCKED (403) — run locally, not from a cloud/server IP: {detail_url}")
        else:
            log.warning(f"HTTP error: {detail_url}: {exc}")
        return None
    except requests.RequestException as exc:
        log.warning(f"Could not fetch {detail_url}: {exc}")
        return None

    resp.encoding = "utf-8"                          #UTF-8 before parsing
    soup = BeautifulSoup(resp.text, "html.parser")

    # Filter 1: Language
    language = extract_table_value(soup, "Jazyk práce").lower()
    if language != LANGUAGE.lower():
        return None

    # Filter 2: Faculty
    faculty = extract_table_value(soup, "Fakulta").lower()
    if faculty.lower() != FACULTY_MATCH.lower():
        return None

    # Filter 3: Submission date
    submitted_raw = extract_table_value(soup, "Datum podání práce")
    submitted_dt  = parse_czech_date(submitted_raw)
    if submitted_dt is None or submitted_dt < SUBMITTED_AFTER:
        return None

    title_eng    = extract_table_value(soup, "Název práce").lower()
    programme    = extract_table_value(soup, "Studijní program").lower()
    department   = extract_table_value(soup, "Katedra").lower()
    defended_raw = extract_table_value(soup, "Datum obhajoby").lower()
    abstract     = extract_table_value(soup, "Abstrakt").lower()
    key_words    = extract_table_value(soup, "Klíčová slova").lower()

    # PDF link — extract thesis ID and build direct download URL
    pdf_url = None
    for link in soup.find_all("a", href=True):
        href = link["href"]
        m = re.search(r"insis\.vse\.cz/zp/(\d+)$", href)
        if m:
            thesis_id = m.group(1)
            pdf_url = f"https://insis.vse.cz/zp/{thesis_id}"
            break

    return {
        "title_eng": title_eng,
        "language":   language,
        "faculty":    faculty,
        "programme":  programme,
        "department": department,
        "submitted":  submitted_raw,
        "defended":   defended_raw,
        "abstract":   abstract,
        "key_words" : key_words,
        "pdf_url":    pdf_url,
    }

## Stage 3 — Download PDFs

In [ ]:
def download_pdf(session: requests.Session, pdf_url: str, dest: Path) -> bool:
    """
    Download a PDF file from a given URL and save it to a local destination.

    The function checks whether the file already exists and skips downloading
    in that case. It streams the content to disk in chunks and verifies that
    the response appears to be a valid PDF (based on Content-Type). If the
    download fails or the content is not a PDF, the function returns False.

    Args:
        session (requests.Session): Active HTTP session used for making requests.
        pdf_url (str): Direct URL to the PDF file.
        dest (Path): Filesystem path where the PDF should be saved.

    Returns:
        bool: True if the file was successfully downloaded or already exists,
              False if the download failed or the content was not a valid PDF.

    Raises:
        None: All exceptions are handled internally. Any request-related errors
        are logged and result in returning False.

    Notes:
        - Uses streamed downloading to handle large files efficiently.
        - Validates the response via the "Content-Type" header to avoid saving
          HTML pages (e.g., login or error pages) as PDFs.
        - If a failure occurs during download, any partially written file is removed.
        - Creates parent directories automatically if they do not exist.
    """

    if dest.exists():
        log.debug(f"Skip (exists): {dest.name}")
        return True
    try:
        with session.get(pdf_url, headers=HEADERS, stream=True,
                        timeout=60, allow_redirects=True) as r:
            r.raise_for_status()
            # Guard: make sure we actually received a PDF, not a login page
            content_type = r.headers.get("Content-Type", "")
            if "pdf" not in content_type.lower() and "octet" not in content_type.lower():
                log.warning(f"Not a PDF ({content_type}): {pdf_url} — skipping")
                return False
            dest.parent.mkdir(parents=True, exist_ok=True)
            with open(dest, "wb") as f:
                for chunk in r.iter_content(chunk_size=8192):
                    f.write(chunk)
        return True
    except requests.RequestException as exc:
        log.warning(f"Download failed {pdf_url}: {exc}")
        if dest.exists():
            dest.unlink()
        return False

## Main — run all stages

In [24]:
DOWNLOAD_DIR.mkdir(parents=True, exist_ok=True)

with requests.Session() as session:

    # Stage 1 — collect all listing URLs
    raw_entries = collect_all_listings(session)
    # De-duplicate
    seen, unique = set(), []
    for e in raw_entries:
        if e["detail_url"] not in seen:
            seen.add(e["detail_url"])
            unique.append(e)
    log.info(f"Unique thesis pages to inspect: {len(unique)}")

    # Stage 2 — filter by faculty / language / submission date
    log.info(
        f"Filters → faculty='{FACULTY_MATCH}' | "
        f"language='{LANGUAGE}' | submitted>={SUBMITTED_AFTER.date()}"
    )
    qualified = []
    for entry in tqdm(unique, desc="Inspecting detail pages"):
        details = fetch_thesis_details(session, entry["detail_url"])
        if details:
            entry.update(details)
            qualified.append(entry)
            log.info(f"  ✓ [{len(qualified):3d}] {entry['title'][:65]} ({entry['submitted']})")
        polite_sleep()

    log.info(f"Qualified (FIS + English + 2020-present): {len(qualified)}")

    # Save metadata CSV
    df = pd.DataFrame(qualified)
    df.to_csv(METADATA_CSV, index=False, encoding="utf-8-sig")
    log.info(f"Metadata saved → {METADATA_CSV}")

    # Stage 3 — download PDFs 
    downloadable = [t for t in qualified if t.get("pdf_url")]
    skipped = len(qualified) - len(downloadable)
    if skipped:
        log.warning(f"{skipped} thesis(es) had no PDF link — skipped.") # If the file could not be retrieved it is skipped

    success = 0
    for thesis in tqdm(downloadable, desc="Downloading PDFs"):
        year_prefix = thesis["submitted"][-4:] if thesis.get("submitted") else "xxxx"
        safe_name   = sanitize_filename(f"{year_prefix}_-_{thesis['author']}_-_{thesis['title_eng']}")
        dest = DOWNLOAD_DIR / f"{safe_name}.pdf"
        if download_pdf(session, thesis["pdf_url"], dest):
            success += 1
        polite_sleep()

log.info(f"Finished. {success}/{len(downloadable)} PDFs saved to '{DOWNLOAD_DIR}/'.")
log.info(f"Full index: '{METADATA_CSV}'")

12:34:13 [INFO] Year 2020: 30 listing pages found.


Listing 2020:   0%|          | 0/30 [00:00<?, ?it/s]

12:34:16 [INFO] Year 2021: 33 listing pages found.


Listing 2021:   0%|          | 0/33 [00:00<?, ?it/s]

12:34:19 [INFO] Year 2022: 34 listing pages found.


...

Listing 2026:   0%|          | 0/38 [00:00<?, ?it/s]

12:34:36 [INFO] Raw entries collected: 12372
12:34:36 [INFO] Unique thesis pages to inspect: 12372
12:34:36 [INFO] Filters → faculty='Fakulta informatiky a statistiky' | language='English' | submitted>=2020-01-01


Inspecting detail pages:   0%|          | 0/12372 [00:00<?, ?it/s]

12:34:46 [INFO]   ✓ [  1] Implementace knihovny pro použití Domain-Driven Designu v prostře (6. 5. 2020)
12:34:48 [INFO]   ✓ [  2] Model-driven development a jeho využití při vývoji velkých inform (10. 5. 2020)
12:34:50 [INFO]   ✓ [  3] Návrh uživatelského rozhraní pro chytré brýle (8. 5. 2020)
12:34:50 [INFO]   ✓ [  4] Serverless Computing: Benefits and challenges (11. 5. 2020)
12:34:50 [INFO]   ✓ [  5] Obchodní hodnota a vliv principů UX a UI v internetovém obchodě (10. 5. 2020)
12:34:51 [INFO]   ✓ [  6] Analyza využití Google Analytics v konkrétní firmě (11. 5. 2020)
12:34:53 [INFO]   ✓ [  7] Prehľad psychosociálnych a kognítivnych faktorov ktoré ovplyvňujú (11. 5. 2020)
12:34:54 [INFO]   ✓ [  8] Světlo jako prvek vizuálního vyprávění (10. 5. 2020)
12:34:55 [INFO]   ✓ [  9] Possibilities of Amazon AWS and Google GCP cloud services in spec (11. 5. 2020)
12:34:55 [INFO]   ✓ [ 10] Role režiséra v reklamní tvorbě a jeho možnosti kreativně ovlivni (11. 5. 2020)
12:34:58 [INFO]   ✓ [ 11] 

12:45:29 [INFO] Finished. 120/120 PDFs saved to 'downloaded_theses/'.
12:45:29 [INFO] Full index: 'theses_metadata.csv'


# Data Scraped - Summary 
Scraper for data was used 27.05.2026. Totally there were 162 documents retrieved based on our restrictions for better analysis and "Strategic plan for Artificial Intelligence
and Machine Learning" that Vice-Dean of reaserch requested. 
42 out of 162, did not have url for download - manual search on random 5 thesis was conducted. Most are restricted for download because they are from may 2026 and are about to be defended so they are not able to be downloaded yet. Others are not downloadable because of copiright reasons.
So total, there will be 120 pdf files as corpus for further analysis.

This script, also generates metadata, with information about the thesis found that passed the criteria. Main intrested is key words of the thesis and abstract. 

Note: Name of thesis "title" is in submited form and can be in czech (it is perserved bcs it is used as url), "title_eng" is in english and will be used in analysis

Downloaded PDFs have defined structure of saved name, defined by: safe_name = sanitize_filename(f"{year_prefix}\_-\_{thesis['author']}\_-\_{thesis['title_eng']}") - Where year_prefix is just year when the thesis was submited